# RAGAS — Evaluación del sistema RAG de Medicina Basada en la Evidencia (MBE)

## Propósito y alcance
Este cuaderno documenta la evaluación registrada del sistema de recuperación aumentada por generación (RAG) para consultas de Medicina Basada en la Evidencia. Presenta la fuente de datos, la selección de preguntas, la preparación de contextos, la implementación de tres métricas, los resultados por consulta y su interpretación.

La idea es configurar este sistema para poder evaluar dada las diferentes configuraciones y preguntas

## 1. Fuente de datos: registros de consultas
Se utiliza una base de datos SQLite de registros del asistente (`mbe_logs (4).db`), referenciada mediante una ruta del entorno de Google Colab. Primero se inspeccionan las tablas disponibles y luego se carga la tabla `consultas` en un DataFrame.

La ejecución depende de que el archivo exista en la ruta indicada.

In [ ]:
from google.colab import files
import sqlite3
import pandas as pd

# Conectar a la DB
conn = sqlite3.connect('/content/mbe_logs (4).db')

# Ver las tablas disponibles
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tablas)


              name
0        consultas
1  sqlite_sequence


In [ ]:
df = pd.read_sql("SELECT * FROM consultas", conn)

In [ ]:
df.columns

Index(['id', 'timestamp', 'usuario', 'session_id', 'pregunta',
       'chunks_extraidos', 'scores_similitud', 'respuesta',
       'latencia_embedding', 'latencia_retrieval', 'latencia_llm',
       'latencia_total'],
      dtype='object')

## 2. Inspección y selección de las consultas evaluadas
La siguiente consulta exploratoria permite revisar preguntas de un usuario concreto (`docente7`). Después se ajusta la visualización de pandas y se seleccionan los registros con identificadores **3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14 y 15**.


In [ ]:
df[df['usuario'] == 'docente7']['pregunta']

,pregunta
3,¿Cómo se distinguen las revisiones sistemáticas de las revisiones tradicionales?
4,"¿Qué es el ""sesgo de publicación"" y por qué es un problema?"
5,"¿Qué es el ""sesgo de publicación"" y por qué es un problema?"
6,¿Cuáles son algunas preguntas típicas que un paciente o su familia podrían hacer sobre el pronóstico?
7,Cuál es la diferencia clave entre una pregunta sobre daño (harm) y una sobre pronóstico?
8,Cuáles son los cuatro criterios de validez para estudios de precisión diagnóstica?
9,¿Qué es un estándar de referencia (reference standard)?
10,¿Cómo se definen la Sensibilidad y la Especificidad en una tabla de 2x2?
11,¿En qué momento se realizan el ocultamiento de la asignación (allocation concealment) y el cegamiento (blinding)?
12,"¿Cuál es el objetivo de un ensayo de ""no inferioridad"" (non-inferiority trial)?"


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
df_final = df[df['id'].isin([3,4,6,7,8,9,10,11,12,13,14,15])]

In [ ]:
df_final[['pregunta', 'chunks_extraidos', 'respuesta']].head(1)

,pregunta,chunks_extraidos,respuesta
2,Que es la medicina basada en la evidencia y cuales son sus pasos principales?,"[{""id_articulo"": NaN, ""titulo"": NaN}, {""id_articulo"": NaN, ""titulo"": NaN}, {""id_articulo"": NaN, ""titulo"": NaN}]","**Respuesta clara y directa**\n\nLa medicina basada en la evidencia (MBE) es un enfoque que se centra en utilizar la evidencia para tomar decisiones médicas informadas. Los pasos principales de MBE son:\n\n1. **Identificar la mejor evidencia**: Buscar y evaluar la mejor evidencia disponible para resolver el problema o decisión médica.\n2. **Evaluar la confiabilidad de la evidencia**: Determinar la confiabilidad de la evidencia, considerando factores como la calidad de la investigación, la generalización a la población objetivo, etc.\n3. **Considerar múltiples perspectivas**: Considerar múltiples perspectivas y opiniones para tomar una decisión informada.\n4. **Evaluar los riesgos y beneficios**: Evaluar los riesgos y beneficios potenciales de cada opción de tratamiento.\n\n**Pasos específicos**\n\n* Cuando se debe realizar un trial (RCT) con bajo número de eventos, es importante considerar si la evidencia es confiable y si el número de eventos es lo suficientemente grande para generar conclusiones precisas.\n* Si se encuentra una diferencia significativa en los resultados entre los grupos de tratamiento, es importante evaluar si esta diferencia es relevante para el problema o decisión médica en cuestión.\n* Es fundamental considerar la confiabilidad de la evidencia y no confiar"


## 3. Dependencias y modelos de evaluación
Se instala el conjunto de dependencias indicado y se comprueba la disponibilidad de NumPy, Groq y Sentence Transformers. La evaluación usa **`llama-3.3-70b-versatile`** como modelo juez en las métricas que requieren evaluación o generación de texto y **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`** para representar preguntas mediante embeddings.


In [ ]:
!pip install -q "numpy<2.0" sentence-transformers groq

In [ ]:
import numpy as np
print(f"numpy: {np.__version__}")

from groq import Groq
print("groq OK ✓")

from sentence_transformers import SentenceTransformer
print("sentence_transformers OK ✓")

import ast, json, time, pandas as pd
print("Todo listo ✓")

numpy: 1.26.4
groq OK ✓
sentence_transformers OK ✓
Todo listo ✓


In [ ]:
import os
client = Groq(api_key=os.getenv("api_key"))
st_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("Cliente Groq y embeddings listos ✓")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cliente Groq y embeddings listos ✓


## 4. Definición de las métricas
Se calculan tres indicadores entre 0 y 1 mediante funciones propias:

- **Faithfulness (fidelidad):** el modelo juez descompone la respuesta en afirmaciones y marca cuáles están respaldadas por los contextos recuperados. Se calcula la proporción de afirmaciones respaldadas.
- **Answer Relevancy (relevancia de la respuesta):** el modelo genera tres preguntas plausibles a partir de la respuesta. Se calcula la similitud coseno entre el embedding de la pregunta original y los de las preguntas generadas, y se promedia.
- **Context Relevance (relevancia del contexto):** el modelo juez estima el número total de oraciones de los contextos y cuántas considera relevantes para la pregunta. Se calcula la proporción de oraciones relevantes.

Estas son las **definiciones operativas de este cuaderno**; no deben confundirse automáticamente con las implementaciones y versiones oficiales de métricas de la biblioteca RAGAS. Ante errores, las funciones devuelven `None`, que pandas omite por defecto al calcular promedios.


In [ ]:
def calcular_faithfulness(pregunta, contextos, respuesta):
    contexto_str = "\n\n".join(contextos)
    prompt = f"""Eres un evaluador experto. Dado el siguiente contexto y respuesta,
extrae cada afirmación individual de la respuesta y determina si está respaldada por el contexto.

CONTEXTO:
{contexto_str}

RESPUESTA:
{respuesta}

Responde SOLO con un JSON válido así, sin texto adicional:
{{"afirmaciones": [{{"texto": "...", "respaldada": true}}]}}"""

    try:
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw = resp.choices[0].message.content
        raw = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)
        afirmaciones = data["afirmaciones"]
        respaldadas = sum(1 for a in afirmaciones if a["respaldada"])
        return round(respaldadas / len(afirmaciones), 3) if afirmaciones else 0
    except Exception as e:
        print(f"  [faithfulness error] {e}")
        return None


def calcular_answer_relevancy(pregunta, respuesta):
    prompt = f"""Dada la siguiente respuesta, genera 3 preguntas distintas que
alguien podría haber hecho para obtener esa respuesta.

RESPUESTA:
{respuesta}

Responde SOLO con un JSON válido así, sin texto adicional:
{{"preguntas": ["pregunta1", "pregunta2", "pregunta3"]}}"""

    try:
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw = resp.choices[0].message.content
        raw = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)
        preguntas_inversas = data["preguntas"]

        emb_original = st_model.encode(pregunta)
        emb_inversas  = st_model.encode(preguntas_inversas)

        similitudes = []
        for emb in emb_inversas:
            sim = float(
                emb_original @ emb /
                ((emb_original @ emb_original)**0.5 * (emb @ emb)**0.5)
            )
            similitudes.append(sim)

        return round(sum(similitudes) / len(similitudes), 3)
    except Exception as e:
        print(f"  [answer_relevancy error] {e}")
        return None


def calcular_context_relevance(pregunta, contextos):
    contexto_str = "\n\n".join(contextos)
    prompt = f"""Eres un evaluador experto. Dado el siguiente contexto y pregunta,
cuenta el total de oraciones del contexto e identifica cuántas son relevantes
para responder la pregunta.

PREGUNTA:
{pregunta}

CONTEXTO:
{contexto_str}

Responde SOLO con un JSON válido así, sin texto adicional:
{{"total_oraciones": 10, "oraciones_relevantes": 7}}"""

    try:
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw = resp.choices[0].message.content
        raw = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)
        total      = data["total_oraciones"]
        relevantes = data["oraciones_relevantes"]
        return round(relevantes / total, 3) if total > 0 else 0
    except Exception as e:
        print(f"  [context_relevance error] {e}")
        return None

print("Funciones listas ✓")

Funciones listas ✓


## 5. Preparación de los contextos recuperados
Los fragmentos almacenados en `chunks_extraidos` se convierten en listas de texto mediante `extraer_textos`. La función intenta interpretar el contenido como JSON, extrae el campo `texto` de cada fragmento y usa el valor de respaldo `sin contexto` cuando no hay fragmentos válidos o se produce un error.

La nueva columna `contexts_parsed` constituye la entrada de contexto utilizada por las métricas de fidelidad y relevancia del contexto.


In [ ]:
import json
import re

def extraer_textos(chunks_raw):
    try:
        if isinstance(chunks_raw, str):
            # Reemplazar NaN por null para que sea JSON válido
            chunks_str = re.sub(r'\bNaN\b', 'null', chunks_raw)
            chunks = json.loads(chunks_str)
        else:
            chunks = chunks_raw

        textos = []
        for chunk in chunks:
            texto = chunk.get("texto", "")
            if texto and str(texto) != "nan" and texto is not None:
                textos.append(str(texto))

        return textos if textos else ["sin contexto"]

    except Exception as e:
        print(f"Error parseando chunk: {e}")
        return ["sin contexto"]

# Usar .loc para evitar el SettingWithCopyWarning
df_final = df_final.copy()
df_final.loc[:, "contexts_parsed"] = df_final["chunks_extraidos"].apply(extraer_textos)

# Verificar
print(f"Datos listos: {len(df_final)} filas ✓")
print(f"\nEjemplo de contexto parseado (primeros 300 chars):")
print(df_final["contexts_parsed"].iloc[0][0][:300])

Datos listos: 12 filas ✓

Ejemplo de contexto parseado (primeros 300 chars):
sin contexto


## 6. Ejecución registrada de la evaluación por pregunta
Para cada fila seleccionada se calculan las tres métricas a partir de la pregunta, la respuesta y los contextos preparados. Se conserva una fila de resultados por consulta en `df_resultados`. La pausa entre iteraciones está presente en el código original para moderar las solicitudes al servicio.

In [ ]:
resultados = []

for i, row in df_final.iterrows():
    print(f"[{i+1}/{len(df_final)}] {row['pregunta'][:70]}...")

    f  = calcular_faithfulness(row["pregunta"], row["contexts_parsed"], row["respuesta"])
    ar = calcular_answer_relevancy(row["pregunta"], row["respuesta"])
    cr = calcular_context_relevance(row["pregunta"], row["contexts_parsed"])

    resultados.append({
        "pregunta":          row["pregunta"],
        "faithfulness":      f,
        "answer_relevancy":  ar,
        "context_relevance": cr
    })

    print(f"  F={f} | AR={ar} | CR={cr}")
    time.sleep(1.5)  # evitar rate limit de Groq

df_resultados = pd.DataFrame(resultados)
print("\nEvaluación completa ✓")

[3/12] Que es la medicina basada en la evidencia y cuales son sus pasos princ...
  F=0.0 | AR=0.807 | CR=0
[4/12] ¿Cómo se distinguen las revisiones sistemáticas de las revisiones trad...
  F=1.0 | AR=0.904 | CR=0.444
[6/12] ¿Qué es el "sesgo de publicación" y por qué es un problema?...
  F=1.0 | AR=0.586 | CR=0.308
[7/12] ¿Cuáles son algunas preguntas típicas que un paciente o su familia pod...
  F=0.833 | AR=0.87 | CR=0.222
[8/12] Cuál es la diferencia clave entre una pregunta sobre daño (harm) y una...
  F=1.0 | AR=0.87 | CR=0.6
[9/12] Cuáles son los cuatro criterios de validez para estudios de precisión ...
  F=1.0 | AR=0.909 | CR=0.059
[10/12] ¿Qué es un estándar de referencia (reference standard)?...
  F=1.0 | AR=0.719 | CR=0.348
[11/12] ¿Cómo se definen la Sensibilidad y la Especificidad en una tabla de 2x...
  F=1.0 | AR=0.823 | CR=0.0
[12/12] ¿En qué momento se realizan el ocultamiento de la asignación (allocati...
  F=0.714 | AR=0.619 | CR=0.667
[13/12] ¿Cuál es el objetivo d

## 7. Consolidación de resultados y contraste con las metas
El código calcula los promedios de las tres métricas y los compara con los umbrales definidos en el proyecto: **Faithfulness > 0.85**, **Answer Relevancy > 0.80** y **Context Relevance > 0.75**. También muestra el detalle por pregunta.

Los umbrales son criterios del proyecto expresados en el código, no umbrales universales de RAGAS.


In [ ]:
promedios = df_resultados[["faithfulness", "answer_relevancy", "context_relevance"]].mean()

print("=== PROMEDIOS GENERALES ===")
print(promedios.round(3))

print("\n=== METAS DEL PROYECTO ===")
print(f"Faithfulness      {promedios['faithfulness']:.3f}  (meta > 0.85) {'✓' if promedios['faithfulness'] > 0.85 else '✗'}")
print(f"Answer Relevancy  {promedios['answer_relevancy']:.3f}  (meta > 0.80) {'✓' if promedios['answer_relevancy'] > 0.80 else '✗'}")
print(f"Context Relevance {promedios['context_relevance']:.3f}  (meta > 0.75) {'✓' if promedios['context_relevance'] > 0.75 else '✗'}")

# Detalle por pregunta
print("\n=== DETALLE POR PREGUNTA ===")
df_resultados[["pregunta", "faithfulness", "answer_relevancy", "context_relevance"]]

=== PROMEDIOS GENERALES ===
faithfulness         0.823
answer_relevancy     0.801
context_relevance    0.348
dtype: float64

=== METAS DEL PROYECTO ===
Faithfulness      0.823  (meta > 0.85) ✗
Answer Relevancy  0.801  (meta > 0.80) ✓
Context Relevance 0.348  (meta > 0.75) ✗

=== DETALLE POR PREGUNTA ===


,pregunta,faithfulness,answer_relevancy,context_relevance
0,Que es la medicina basada en la evidencia y cuales son sus pasos principales?,0.000,0.807,0.000
1,¿Cómo se distinguen las revisiones sistemáticas de las revisiones tradicionales?,1.000,0.904,0.444
2,"¿Qué es el ""sesgo de publicación"" y por qué es un problema?",1.000,0.586,0.308
3,¿Cuáles son algunas preguntas típicas que un paciente o su familia podrían hacer sobre el pronóstico?,0.833,0.870,0.222
4,Cuál es la diferencia clave entre una pregunta sobre daño (harm) y una sobre pronóstico?,1.000,0.870,0.600
5,Cuáles son los cuatro criterios de validez para estudios de precisión diagnóstica?,1.000,0.909,0.059
6,¿Qué es un estándar de referencia (reference standard)?,1.000,0.719,0.348
7,¿Cómo se definen la Sensibilidad y la Especificidad en una tabla de 2x2?,1.000,0.823,0.000
8,¿En qué momento se realizan el ocultamiento de la asignación (allocation concealment) y el cegamiento (blinding)?,0.714,0.619,0.667
9,"¿Cuál es el objetivo de un ensayo de ""no inferioridad"" (non-inferiority trial)?",1.000,0.852,0.714
